# 0.3 visualize channel map from an Open Ephys recording

**[env: optonpx]**

Parse the channel map actually used in an Open Ephys recording from its `settings.xml`
and plot it for every enabled probe (same layout as notebook 0.2 for `.imro` files).

In [ ]:
# path and params
from pathlib import Path

base_path = Path(r"D:\D1-1-6_IM-1979\ephys\2026-09-17_14-16-02")  # OE recording folder
is_save = False  # True: save one PNG per probe into base_path

In [ ]:
import sys

sys.path.append("..")
from src import oe_parse_folders, oe_parse_xml_params

oe_names, oe_paths = oe_parse_folders(base_path)
if oe_paths["xml"] is None:
    raise FileNotFoundError(f"settings.xml not found under: {oe_paths['node']}")
probes_params, adc_params = oe_parse_xml_params(oe_paths["xml"])
print(f"settings.xml: {oe_paths['xml']}")
print(f"{len(probes_params)} enabled probe(s): {[p['stream_name'] for p in probes_params]}")

In [ ]:
# per-probe summary
for probe in probes_params:
    print(f"=== {probe['stream_name']} ===")
    print(f"  probe name : {probe['probe_name']}")
    print(f"  part / SN  : {probe['part_number']} / {probe['serial_number']}")
    print(f"  port / dock: {probe['port']} / {probe['dock']}")
    print(f"  reference  : {probe['reference']}")
    print(f"  channels   : {len(probe['channel_electrode'])}")
    for shank in sorted(set(probe["channel_shank"].values())):
        chs = [ch for ch, sh in probe["channel_shank"].items() if sh == shank]
        banks = sorted(set(probe["channel_bank"][ch] for ch in chs))
        ys = [probe["channel_ypos"][ch] for ch in chs]
        print(f"  shank {shank}: {len(chs):3d} channels, banks {banks}, "
              f"y {min(ys):5d}-{max(ys):5d} um")

In [ ]:
# option 1: show figure inline, using svg format to improve sharpness
%matplotlib inline
import matplotlib.pyplot as plt
%config InlineBackend.figure_format = 'svg'
# option 2: plot in separate figure window, with tk or qt backend
# %matplotlib tk
# %matplotlib qt
from src import plot_electrode_map

for probe in probes_params:
    stream_name = probe["stream_name"]
    fig = plot_electrode_map(probe["channel_electrode"], title=f"{base_path.name} {stream_name}")
    if is_save:
        fig_path = base_path / f"{base_path.name}_{stream_name}_channel_map.png"
        fig.savefig(fig_path, bbox_inches="tight", dpi=200)
        print(f"Saved: {fig_path}")
    plt.show()